In [ ]:
# Post-hoc Dunn test for significant indicators
sig_indicators = [r['indicator'] for r in results if r['p'] < 0.05]

for ind in sig_indicators:
    print(f"\n{'='*50}")
    print(f"  Dunn post-hoc: {INDICATOR_LABELS[ind]}")
    print(f"{'='*50}")
    dunn = sp.posthoc_dunn(df, val_col=ind, group_col='regime_iconocratico', p_adjust='bonferroni')
    print(dunn.round(4))

# Kruskal-Wallis on composite score
print(f"\n{'='*50}")
print(f"  Kruskal-Wallis: Score composto de ENDURECIMENTO")
print(f"{'='*50}")
samples = [groups[r]['purificacao_composto'].dropna().values for r in regimes]
H, p = stats.kruskal(*samples)
print(f"H = {H:.2f}, p = {p:.4f}")
if p < 0.05:
    dunn_comp = sp.posthoc_dunn(df, val_col='purificacao_composto', 
                                 group_col='regime_iconocratico', p_adjust='bonferroni')
    print("\nDunn post-hoc (Bonferroni):")
    print(dunn_comp.round(4))

# 02 — Regimes × Morfologia: Kruskal-Wallis (Cap. 6.2)

Testa se os indicadores de purificação diferem significativamente
entre regimes iconocráticos (fundacional, normativo, militar).

**Teste:** Kruskal-Wallis H (não-paramétrico, dados ordinais)

**Post-hoc:** Dunn com correção de Bonferroni

**H₀:** A distribuição de cada indicador é igual entre os três regimes.

**H₁:** Ao menos um regime difere significativamente.

In [ ]:
import pandas as pd
import numpy as np
from scipy import stats
import scikit_posthocs as sp
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid", font_scale=1.1)

df = pd.read_csv('../data/processed/corpus_dataset.csv')

INDICATORS = ['desincorporacao', 'rigidez_postural', 'dessexualizacao',
              'uniformizacao_facial', 'heraldizacao', 'enquadramento_arquitetonico',
              'apagamento_narrativo', 'monocromatizacao', 'serialidade', 'inscricao_estatal']

INDICATOR_LABELS = {
    'desincorporacao': 'Desincorporação', 'rigidez_postural': 'Rigidez postural',
    'dessexualizacao': 'Dessexualização', 'uniformizacao_facial': 'Uniformização facial',
    'heraldizacao': 'Heraldicização', 'enquadramento_arquitetonico': 'Enquadramento arq.',
    'apagamento_narrativo': 'Apagamento narrativo', 'monocromatizacao': 'Monocromatização',
    'serialidade': 'Serialidade', 'inscricao_estatal': 'Inscrição estatal',
}

regimes = ['fundacional', 'normativo', 'militar']
groups = {r: df[df.regime_iconocratico == r] for r in regimes}

print(f"N por regime: {', '.join(f'{r}={len(g)}' for r, g in groups.items())}")
print(f"Total: {len(df)}")
print(f"\n{'='*70}")
print(f"{'Indicador':<25s} {'H':>8s} {'p-value':>10s} {'Sig.':>6s} {'η²':>8s}")
print(f"{'='*70}")

results = []
for ind in INDICATORS:
    samples = [g[ind].dropna().values for g in groups.values()]
    H, p = stats.kruskal(*samples)
    # Effect size: eta-squared for Kruskal-Wallis
    n = sum(len(s) for s in samples)
    eta2 = (H - len(regimes) + 1) / (n - len(regimes))
    sig = '***' if p < 0.001 else '**' if p < 0.01 else '*' if p < 0.05 else 'ns'
    results.append({'indicator': ind, 'H': H, 'p': p, 'sig': sig, 'eta2': eta2})
    print(f"{INDICATOR_LABELS[ind]:<25s} {H:8.2f} {p:10.4f} {sig:>6s} {eta2:8.3f}")

print(f"{'='*70}")
sig_count = sum(1 for r in results if r['p'] < 0.05)
print(f"\n{sig_count}/10 indicadores com diferença significativa (p < 0.05)")

## 2.2 Post-hoc Dunn para indicadores significativos